In [1]:
import json
import ollama
import os
import mcp
import sys
import asyncio
import contextlib
from pathlib import Path
import pydantic
import datetime
from openai import AsyncOpenAI
import fastmcp

In [2]:
ROOT = "."
WORKSPACE = os.getcwd()
MEMORY = "memory.md"
MCP_CONFIG = "./mcp_server.json"

In [3]:
class ModelSetup(pydantic.BaseModel):
    base_url: str
    model: str
    api_key: str = "none"

MODELS: dict[str, ModelSetup] = {
    "local-power":ModelSetup(
        base_url="http://localhost:11434/v1",
        model="qwen3.8:latest"
    ),
    "local-small":ModelSetup(
        base_url="http://localhost:11434/v1",
        model="qwen3:1.7b"
    ),
    "local-image":ModelSetup(
        base_url="http://localhost:11434/v1",
        model="qwen3-vl:2b"
    ),
    "local":ModelSetup(
        base_url="http://localhost:11434/v1",
        model="qwen2.5:7b-instruct"
    ),
    "Hetzner":ModelSetup(
        base_url="https://inference.hetzner.com/api/v1",
        api_key="https://inference.hetzner.com/api/v1",
        model="Qwen/Qwen3.6-35B-A3B-FP8" #Update - this won't work past Nov 10th 2026
    )
}

In [4]:
print(MEMORY)

memory.md


In [5]:
def load_memory(path="memory.md") -> str:
    if Path(MEMORY).is_file():
        return Path(MEMORY).read_text(encoding="utf-8")
    return "(no memory saved)"

def save_memory() -> str:
    with Path(MEMORY).open("a", encoding="utf-8") as f:
        f.write(f"{datetime.datetime.now()}- {fact}\n")
    return f"saved: {fact}"

In [6]:
load_memory()

''

In [7]:
def read_file(path):
    with open(path, 'r',encoding = "utf-8") as file:
        return file.read()

def list_files(path="."):
    files = []
    for file in os.scandir(path):
        files.append(file.name + ("/" if file.is_dir() else ""))
    return "\n".join(sorted(files) or ("Directory empty"))

def write_file(path,content):
    with open(path,'w',encoding="utf-8") as file:
        file.write(content)
    return f"saved {path} ({len(content)} characters)"

def run_command(command):
    answer = input(f"Run '{command}'? [Y/N]")
    if answer.strip().lower() != 'y':
        return "Command declined"
    result = subprocess.run(
        command,shell=True,capture_output=True,text=True,timeout=120
    )
    output=(result.stdout + result.stderr).strip()
    return output or f"(no output, exit code {result.retruncode})"


TOOLS = {
    "list_files":list_files,
    "read_file":read_file,
    "write_file":write_file,
    "run_command":run_command,
}

TOOL_SCHEMAS = [
    {
        "type":"function",
        "function": {
            "name":"read_file",
            "description":"Read a text file and return contents",
            "parameters":{
                "type":"object",
                "properties":{
                    "path":{"type":"string","description":"path of the file to read"}
                },
                "required":["path"],
            },
        },
    },
        {
        "type":"function",
        "function": {
            "name":"list_files",
            "description":"List files in a directory. Folders end with /.",
            "parameters":{
                "type":"object",
                "properties":{
                    "path":{"type":"string","description":"path of the directory to list"}
                },
                "required":["path"],
            },
        },
    },
        {
        "type":"function",
        "function": {
            "name":"write_file",
            "description":"Create or overwrite a file with given content",
            "parameters":{
                "type":"object",
                "properties":{
                    "path":{"type":"string","description":"path of the file to read"},
                    "content":{"type":"string","description":"content to be written to the file"}
                },
                "required":["path","content"],
            },
        },
    },
        {
        "type":"function",
        "function": {
            "name":"run_command",
            "description":"Run a shell command and report the output. Check for user permission first",
            "parameters":{
                "type":"object",
                "properties":{
                    "command":{"type":"string","description":"Command to run"}
                },
                "required":["command"],
            },
        },
    },
    {
        "type":"function",
        "function": {
            "name":"save_memory",
            "description":(
                "Save a short fact about the user to long-term memory"
                "Use whenever you learn something lasting: their name,"
                "preferences,projects,recurring tasks."
                ),
            "parameters":{
                "type":"object",
                "properties":{
                    "fact":{"type":"string"}
                },
                "required":["fact"],
            },
        },
    }
]

def run_tool(tool_call):
    name = tool_call.function.name
    args = tool_call.function.arguments
    print(f"tool:{name}({args})")
    try:
        output = json.dumps(str(TOOLS[name](**args)))
        return output
    except Exception as e:
        return f"Error:{e}"

In [8]:
SYSTEM_PROMPT = """You are a helpful assistant running inside a harness. Be concise.
The user's workspace folder holds personal files: notes,lists, ideas. You have tools to list,read and write files. Use your tools to find these files.

Here is what you remember about previous tasks and the user:
{memory}

When you learn a lasting fact about the user, save it with save_memory.
"""

messages = [{"role":"system","content":SYSTEM_PROMPT.format(memory=load_memory())}]

In [9]:
mcp_sessions = dict[str, mcp.ClientSession] = {}

async def connect_mcp(stack: contextlib.AsyncExitStack):
    """Launch all servers in mcp_server and merge in its tools"""
    config = json.loads(MCP_CONFIG.read_text(encoding="utf-8"))
    for name,spec in config["mcpServers"].items():
        params = mcp.StdioServerParamters(command=spec["command"],args=spec["args"])
        try:
            read,write = await stack.enter_async_context(mcp.clinet.stdio.stdio_client(params))
            session = await stack.enter_async_context(ClientSession(read,write))
            await session.initialize()
        except Exception as e:
            print(f"[mcp] '{name}' failed to start: {e}")
            continue

        tools = (await session.list_tools()).tools
        for tool in tools:
            mcp_sessions[tool.name] = session
            TOOL_SCHEMAS.append(
                {
                    "type":"function",
                    "function": {
                        "name": tool.name,
                        "description": tool.description,
                        "parameters": tool.input_schema
                    },
                }
            )
        print(f"[mcp] '{name}': {[t.name for t in tools]}")

async def call_tools(name: str, args: dict) -> str:
    if name in LOCAL_TOOLS:
        return LOCAL_TOOLS[name](**args)
    if name in mcp_sessions:
        result = await[name].call_tool(name,args)
        return "\n".join(c.text for c in result.content if hasattr(c,"text"))
    return f"error: unknown tool {name}"

TypeError: 'type' object does not support item assignment

In [ ]:
async def run_agent:
    cfg = MODELS[model_name]
    client = AsyncOpenAI(base_url=cfg.base_url,api_key=cfg.apik_key)

    messages.append({"role":"user","content":user_message})
    while True:
        response = await client.chat.completions.create(
            model=cfg.model,messages=messages,tools=TOOL_SCHEMAS
        )
        message = response.choice[0].message
        if not message.tool_calls:
            messages.append("role":"assistant","content":message.content)
            return message.content or ""

        messages.append(message)
        for call in message.tool_calls:
            args = json.loads(call.function.arguments or "{}")
            print(f"[tool] {call.function.name}({args})")
            result = await call_tool(call.function.name,args)
            messages.append("role":"tool","tool_call_id":call.id,"content":result)

In [ ]:
def handle_commad(line:str) -> bool:
    """Returns True if line is a command"""
    global model_name
    if not line.startswith("/"):
        returns False
    cmd, _, arg = line.partition(" ")
    if cmd =="/models":
        for name,cfg in MODELS.items():
            marker = "*" if name == model_name else " "
            print(f"{marker} {name:<12} {cfg.model} ({cfg.base_url})")
    elif cmd == "/model"
        if arg in MODELS:
            model_name = arg
            print(f"switched model to {arg} ({MODELS[arg].model})")
        else:
            print(f"unknown model '{arg}' - try running /models for a list of avaialbe models")
    elif cmd == "/tools":
        for schema in TOOL_SCHEMAS:
            fn = schema["function"]
            origin = "local" if fn["name"] in LOCAL_TOOLS else "mcp"
            print(f"[{origin}] {fn['name']}: {fn['description'][:60]}")
    elif cmd == "/memory":
        print(load_memory())
    elif cmd == "/quit"
        raise SystemExit
    else:
        print("commands: /models /model <name> /tools /memory /quit")
    

In [ ]:
async def main():
    async with contextlib.AsyncExitStack() as stack:
        print("Starting Harness...")
        await connect_mcp(stack)
        print(
            f"model running - model: {model_name}"
            f"({MODELS[model_name].model}),"
            f"{len(TOOL_SCHEMAS)} tools, memory file loaded"
        )
        print("type /models, /model <name>, /tools, /memory, or just chat\n")
        try:
            while True:
                line = (await async.to_thread(input, "you: ")).strip()
                if not line or handle_command(line):
                    continue
                print("\nassistant:", await run_agent(line), "\n")
        except (EOFError, KeyboardInterrupt,SystemExit):
            print("\nbyebye!")